In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from langchain_groq import ChatGroq 
from langchain_core.messages import HumanMessage,  SystemMessage
from langchain_core.prompts import ChatPromptTemplate

def main():
        model = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0
    )
        print("Approach 1: messages")
        
        messages = [
                SystemMessage(content="You are a helful translator"),
                HumanMessage(content="Translate 'Hello, world' to French")
        ]   

        message_response = model.invoke(messages)
        print("Message response: ", message_response.content)
        print("\n")


        print("Approach 2: Templates")

        template = ChatPromptTemplate.from_messages([
                ("system", "you are a helpful translator"),
                ("human", "Translate {text} to {language}")
        ])

        template_chain = template | model
        template_response = template_chain.invoke({
                "text" : "Hello word",
                "language" : "French"
        })

        print("Template response: ",template_response.content)


if __name__ == "__main__":
        main()


Approach 1: messages
Message response:  **French:** « Bonjour, le monde »


Approach 2: Templates
Template response:  The English word **“Hello”** translates to **“Bonjour”** in French.


### Example 2: Dynamic Message Construction

Learn how to build message arrays programmatically and use few-shot prompting with messages.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from typing import List


def create_conversation(role: str, examples: List[dict], new_question: str) -> list[BaseMessage]:
    """Build message arrays programatically"""
    messages: list[BaseMessage] = [SystemMessage(content=f"You are a {role}")]

    for example in examples:
        messages.append(HumanMessage(content=example["question"]))
        messages.append(AIMessage(content=example["answer"]))

    messages.append(HumanMessage(content=new_question)) 

    return messages   


def main():
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0
    )

    # Few-Shot Learning with Messages
    emogi_messages = create_conversation(
        "emogi translator",
        [
            {"question": "happy", "answer" : "😊"},
            {"question" : "sad", "answer" : "😢"},
            {"question" : "excied", "answer" : "🎉"}
        ],
        "surprised"
    )

    print(f"Messages constructed: {len(emogi_messages)}")
    llm_response = llm.invoke(emogi_messages)
    print("AI Message: ", llm_response.content)

if __name__ == "__main__":
    main()
    

Messages constructed: 8
AI Message:  😲


### Example 3: Basic Templates

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_groq import ChatGroq

def main():
    template1 = ChatPromptTemplate.from_messages([
        ("system" , "You are a helpful assistant that translates {input_language} to {output_language}"),
        ("human" , "{text}")
    ])
    

    model_3 = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0
    )

    chain = template1 | model_3

    french_response = chain.invoke({
        "input_language"   : "English",
        "output_language"  : "French",
        "text"             : "how are you!"
    })
    print("French : ", french_response.content)


    spanish_response = chain.invoke({
        "input_language" : "English",
        "output_language" : "Spanish",
        "text" : "How are you doing"
    })
    print("Spanish: ", spanish_response.content)


    response_urdu = chain.invoke({
        "input_language" : "English",
        "output_language" : "Urdu",
        "text" : "How are you doing"
    })
    print("Urdu: ",response_urdu.content)


    response_pashto = chain.invoke({
        "input_language" : "English",
        "output_language" : "Pashto",
        "text" : "How are you doing"
    })
    print("Pashto: ",response_pashto.content)    



if __name__ == "__main__":
    main()


French :  Comment ça va !
Spanish:  ¿Cómo estás?
Urdu:  آپ کیسے ہیں؟
Pashto:  ته څنګه یې؟


### Example 4: Template Formats

In [19]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate


def main():
   
    model4 = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0
    )

    chat_template = ChatPromptTemplate.from_messages([
        ("system" , "You are a helpful {role}"),
        ("human" , "{question}")
    ])

    chat_chain = chat_template | model4

    result1 = chat_chain.invoke({
        "role" : "mathematics tutor",
        "question" : "what 2+2?"
    })     

    print("ChatPromptTemplate result:", result1.content)


    simple_template = PromptTemplate.from_template(
        "Answer this {topic} question briefly: {question}"
    )   

    simple_chain = simple_template | model4

    result2 = simple_chain.invoke({
        "topic" : "Biology",
        "question" : "what is enzymes?"
    })    

    print("simple template answer: ", result2.content)


if __name__ == "__main__":
    main()


ChatPromptTemplate result: 2 + 2 = 4.
simple template answer:  Enzymes are biological catalysts—typically proteins—that speed up chemical reactions in living organisms by lowering the activation energy required for the reaction, without being consumed in the process. They bind specific substrates at their active sites, convert them into products, and can be regulated to control metabolic pathways.


### Example code: Few-shot chat prompt template

In [21]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

def main():
    model5 = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0
    )    


    examples = [
        {"input" : "happy", "output" : "😊"},
        {"input" : "sad", "output" : "😢"},
        {"input" : "excited", "output" : "🎉"}
    ]

    example_template = ChatPromptTemplate.from_messages([
        ("human", "{input}"),
        ("ai" , "{output}")
    ])

    few_shot_template = FewShotChatMessagePromptTemplate(
        example_prompt=example_template,
        examples=examples
    )


    final_template = ChatPromptTemplate.from_messages([
        ("system", "You are an emogi translator.Convert the word to emogi"),
        few_shot_template,
        ("human", "{input}")
    ])


    chain = final_template | model5


    for word in ["angry", "confused","Love"]:
        result = chain.invoke({"input": word})
        print(f"{word} -> {result.content}")


if __name__ == "__main__":
    main()

angry -> 😠
confused -> 😕
Love -> ❤️


### Example 6: Template Composition

In [23]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

def main():
    model6 = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0
    )    

    base_instruction = [
        """You are a  {role} assistant.Your communication syle is {style}.
        Always be helpful and professional"""
         ]

    educator_template = ChatPromptTemplate.from_messages([
        ("system", base_instruction),
        ("system", "Focus on teaching concepts clearly with examples"),
        ("human", "{question}")
    ])


    chain = educator_template | model6

    result_chain = chain.invoke({
        "role" : "python programming",
        "style" : "friendly and encouraging",
        "question" : "what is list comprehension in short"
    })

    print("Educator Response:")
    print(result_chain.content)

   
if __name__ == "__main__":
    main()

Educator Response:
**List comprehension** is a compact way to create a new list by applying an expression to each item in an existing iterable (like a list, range, or string) and optionally filtering items with a condition.

**Basic syntax**

```python
new_list = [ expression for item in iterable if condition ]
```

- `expression` – what you want each element of the new list to look like (often just `item` itself).
- `for item in iterable` – loops over the original collection.
- `if condition` – (optional) keeps only the items that satisfy the condition.

**Simple example**

```python
# Get the squares of the numbers 0‑4
squares = [x**2 for x in range(5)]
print(squares)   # Output: [0, 1, 4, 9, 16]
```

**With a filter**

```python
# Keep only the even numbers and double them
evens_doubled = [x*2 for x in range(10) if x % 2 == 0]
print(evens_doubled)   # Output: [0, 4, 8, 12, 16]
```

List comprehensions are concise, readable, and often faster than building a list with a traditional `f